In [1]:
#!/usr/bin/env python
# coding: utf-8

import os, sys, pickle
import jax, jaxlib
from pathlib import Path
import jax.numpy as jnp
import matplotlib.pyplot as plt
import scipy
from scipy.io import loadmat
import numpy as np
import h5py
import argparse

import torch
import flax
from flax import linen as nn
import optax
from sklearn.model_selection import train_test_split
from typing import Callable, Sequence

from tqdm import tqdm
from utils_jax import *

In [2]:
# seed = 42
seed = 3538
# seed = np.random.choice(np.arange(99999), size=1, replace=True)[0]
np.random.seed(seed)
key = jax.random.PRNGKey(seed)
print(f"Seed set with value = {seed}")

Seed set with value = 3538


In [3]:
base_path = "/home/dnayak2/scr4_sgoswam4/Dibya/backup/Datasets/3D_Heat_Conduction/"

outputs = jnp.load(os.path.join(base_path, "processed/3d_heat_field_results.npz"))['heat_field']

Ns, Nt, Nx, Ny, Nz = outputs.shape
print(f"Ns: {Ns}, Nt: {Nt}, Nx: {Nx}, Ny: {Ny}, Nz: {Nz}")

#Get the xspan and yspan
xspan = jnp.linspace(0, 1, Nx)
yspan = jnp.linspace(0, 1, Ny)
zspan = jnp.linspace(0, 1, Nz)
tspan = jnp.linspace(0, 1, Nt)

#Print the xspan and yspan
print(f"tspan: {tspan.shape}, xspan: {xspan.shape}, yspan: {yspan.shape}, zspan: {zspan.shape}")

Ns: 1000, Nt: 101, Nx: 32, Ny: 32, Nz: 16
tspan: (101,), xspan: (32,), yspan: (32,), zspan: (16,)


In [4]:
#Compute dt
dt_val = (1-0)/(Nt-1)
print(f"Computed dt: {dt_val}")

tt = int(Nt//3)
print(f"End timestep of training: {tt}")

#Creating the input and output training data
init_timestep = 0
end_timestep = tt

input_data_NN = outputs[:, init_timestep, :, :, :]    
output_data_NN = outputs[:, init_timestep+1, :, :, :]

for i in range(init_timestep+1, end_timestep):
    input_data_NN = jnp.vstack((input_data_NN, outputs[:,i,:,:,:]))
    output_data_NN = jnp.vstack((output_data_NN, outputs[:,i+1,:,:,:]))
print("After data arrangement shapes of inputs and outputs:")
print(f"Inputs: {input_data_NN.shape}, Outputs: {output_data_NN.shape}")

#Reshaping the output_data_NN from (ns*nt//2, nx, ny, nz) to (ns*nt//2, nx*ny*nz)
#Input_data_NN remains as it is, i.e., (ns*nt//2, nx, ny, nz)
output_data_NN = output_data_NN.reshape(output_data_NN.shape[0], 
                                        output_data_NN.shape[1]*output_data_NN.shape[2]*output_data_NN.shape[3])
print(input_data_NN.shape, output_data_NN.shape)


input_data_NN_train, input_data_NN_test, output_data_NN_train, output_data_NN_test = \
                        train_test_split(input_data_NN, output_data_NN, test_size = 0.2, random_state = 42)
print(input_data_NN_train.shape, input_data_NN_test.shape, 
      output_data_NN_train.shape, output_data_NN_test.shape)

#Freeing memory by deleting input_data_NN and output_data_NN
del input_data_NN, output_data_NN

Computed dt: 0.01
End timestep of training: 33
After data arrangement shapes of inputs and outputs:
Inputs: (33000, 32, 32, 16), Outputs: (33000, 32, 32, 16)
(33000, 32, 32, 16) (33000, 16384)
(26400, 32, 32, 16) (6600, 32, 32, 16) (26400, 16384) (6600, 16384)


In [5]:
class branch_net(nn.Module):

    layer_sizes: Sequence[int] 
    activation: Callable
    
    @nn.compact
    def __call__(self, x):
        init = nn.initializers.glorot_normal()
        
        # #x has shape (ns, nx, ny, nz) - so add channel dimension: (ns, nx, ny, nz, nc)
        x = x[..., jnp.newaxis]
        
        #Convolutional layers
        x = nn.Conv(features = 32, kernel_size = (3,3,3), strides = (2,2,1), padding = "SAME")(x)
        x = nn.activation.gelu(x)
        x = nn.max_pool(x, window_shape=(2,2,2), strides = (2,2,2), padding = "SAME")
        
        x = nn.Conv(features = 32, kernel_size = (2,2,2), strides = (2,2,1), padding = "SAME")(x)
        x = nn.activation.gelu(x)
        x = nn.avg_pool(x, window_shape = (2,2,2), strides = (2,2,2), padding = "SAME")
        
        x = x.flatten()   #flatten
        
        #MLP layers
        for i, layer in enumerate(self.layer_sizes[:-1]):
            x = nn.Dense(layer, kernel_init = init)(x)
            x = self.activation(x)
        x = nn.Dense(self.layer_sizes[-1], kernel_init = init)(x)
        return x

class trunk_net(nn.Module):
    trunk_layer_config: Sequence[int]
    activation: Callable
    
    @nn.compact
    def __call__(self, x):
        
        init = nn.initializers.glorot_normal()
        
        #Trunk network forward pass
        for i, layer_size in enumerate(self.trunk_layer_config):
            x = nn.Dense(layer_size, kernel_init = init)(x)
            x = self.activation(x)
        return x

def add_fourier_features(inputs, num_frequencies=10, max_freq=10):
    x = inputs[:, 0:1]
    y = inputs[:, 1:2]
    z = inputs[:, 2:]
    
    freqs = jnp.pi * jnp.linspace(1, max_freq, num_frequencies).reshape(1, -1)
    
    x_feat = jnp.concatenate([jnp.sin(freqs * x), jnp.cos(freqs * x)], axis=-1)
    y_feat = jnp.concatenate([jnp.sin(freqs * y), jnp.cos(freqs * y)], axis=-1)
    z_feat = jnp.concatenate([jnp.sin(freqs * z), jnp.cos(freqs * z)], axis=-1)
    
    return jnp.concatenate([inputs, x_feat, y_feat, z_feat], axis=-1)

class WaveletAct(nn.Module):
    init_w1: float = 1.0
    init_w2: float = 1.0
    init_omega: float = 1.0

    @nn.compact
    def __call__(self, x):
        w1 = self.param('w1', nn.initializers.constant(self.init_w1), (1,))
        w2 = self.param('w2', nn.initializers.constant(self.init_w2), (1,))
        omega = self.param('omega', nn.initializers.constant(self.init_omega), (1,))
        return w1 * jnp.sin(omega * x) + w2 * jnp.cos(omega * x)

class DeepONet(nn.Module):

    branch_net_config: Sequence[int]
    trunk_net_config: Sequence[int]
    use_Fourier_feat: bool = False

    def setup(self):
        self.branch_net = branch_net(self.branch_net_config, nn.activation.gelu)
        self.trunk_net = trunk_net(self.trunk_net_config, WaveletAct())

    @nn.compact
    def __call__(self, x_branch, x_trunk):

        if self.use_Fourier_feat:
            #Encode x_trunk into fourier features
            x_trunk = add_fourier_features(x_trunk)
        
        #Vectorize over multiple samples of input functions
        branch_outputs = jax.vmap(self.branch_net, in_axes = 0)(x_branch)
        
        #Vectorize over multiple query points
        trunk_outputs = jax.vmap(self.trunk_net, in_axes = 0)(x_trunk)
        bias = self.param('bias', nn.initializers.zeros, (1,))      
        
        inner_product = jnp.einsum('ik,jk->ij', branch_outputs, trunk_outputs)
        inner_product+=bias
        return inner_product


#Form branch and trunk inputs train

#Create for trunk network
[x,y,z] = jnp.meshgrid(xspan, yspan, zspan, indexing = 'ij')
grid = jnp.transpose(jnp.array([x.flatten(), y.flatten(), z.flatten()]))
print(grid.shape)

#Creating the training data for branch and trunk inputs
branch_inputs_train = input_data_NN_train
trunk_inputs_train = grid
outputs_train = output_data_NN_train

print("Shape of branch inputs train: ",branch_inputs_train.shape)
print("Shape of trunk inputs train: ",trunk_inputs_train.shape)
print("Shape of outputs train: ",outputs_train.shape)
print("Shape of grid: ",grid.shape)


#For branch and trunk inputs test
branch_inputs_test = input_data_NN_test
trunk_inputs_test = grid
outputs_test = output_data_NN_test

print("Shape of branch inputs test: ",branch_inputs_test.shape)
print("Shape of trunk inputs test: ",trunk_inputs_test.shape)
print("Shape of outputs test: ",outputs_test.shape)
print("Shape of grid: ",grid.shape)

#DeepONet settings
num_sensor_locations = branch_inputs_train.shape[1]
num_query_locations = 3
latent_vector_size = 100

branch_network_layer_sizes = [256, 128] + [latent_vector_size]
trunk_network_layer_sizes = [128]*4 + [latent_vector_size]

model = DeepONet(branch_network_layer_sizes, trunk_network_layer_sizes)
model_fn = jax.jit(model.apply)
print(f"Model: {model}")

(16384, 3)
Shape of branch inputs train:  (26400, 32, 32, 16)
Shape of trunk inputs train:  (16384, 3)
Shape of outputs train:  (26400, 16384)
Shape of grid:  (16384, 3)
Shape of branch inputs test:  (6600, 32, 32, 16)
Shape of trunk inputs test:  (16384, 3)
Shape of outputs test:  (6600, 16384)
Shape of grid:  (16384, 3)
Model: DeepONet(
    # attributes
    branch_net_config = [256, 128, 100]
    trunk_net_config = [128, 128, 128, 128, 100]
    use_Fourier_feat = False
)


In [6]:
#Freeing memory by deleting inputs and outputs

# Initialize model parameters
key = jax.random.PRNGKey(seed)   #42

training_loss_history = []
test_loss_history = []
num_epochs = int(1.5e5)
batch_size = 64

min_test_loss = jnp.inf

filepath = Path(f'DeepONet_NODE/TI_DON_16767650_seed3538', 
                parents=True, exist_ok=True)
filename = f"model_params_best_{seed}.pkl"

In [7]:
# @jax.jit
def inference_ab(u_curr, u_prev, trunk_inputs_test, dt):
    
    # Step 1: Apply the predictor (Adams-Bashforth) using u_curr and u_prev
    u_dot_curr = model_fn(best_params, u_curr, trunk_inputs_test)  # Predict the rate of change at u_curr
    u_dot_prev = model_fn(best_params, u_prev, trunk_inputs_test)  # Predict the rate of change at u_prev
    
    #Reshaping u_dot_curr and u_dot_prev to broadcast compatible with u_curr
    u_dot_curr = u_dot_curr.reshape(u_dot_curr.shape[0], Nx, Ny, Nz)
    u_dot_prev = u_dot_prev.reshape(u_dot_prev.shape[0], Nx, Ny, Nz)
    
    # Adams-Bashforth predictor (using previous two points)
    u_pred = u_curr + dt * (1.5 * u_dot_curr - 0.5 * u_dot_prev)
    
    # Step 2: Apply the corrector (Adams-Moulton) using the predicted u_pred
    u_dot_pred = model_fn(best_params, u_pred, trunk_inputs_test)  # Predict the rate of change at u_pred
    
    #Reshaping u_dot_pred to broadcast compatible with u_curr, u_dot_curr, u_dot_prev
    u_dot_pred = u_dot_pred.reshape(u_dot_pred.shape[0], Nx, Ny, Nz)
    
    # Adams-Moulton corrector (refine the prediction using u_pred)
    u_next = u_curr + dt * (5/12 * u_dot_pred + 8/12 * u_dot_curr - 1/12 * u_dot_prev)
    
    return u_next


# @jax.jit
def inference_rk(u_curr, trunk_inputs_test, dt):
    
    # Predict the system dynamics (u_dot) at the current state using the model
    u_dot = model_fn(best_params, u_curr, trunk_inputs_test)  # Model's predicted rate of change

    # Implementing the 4th-order Runge-Kutta (RK4) time-stepping method
    k1 = u_dot   #(ns*nt, nx*ny)
    k1 = k1.reshape(k1.shape[0], Nx, Ny, Nz)   #(ns*nt, nx, ny)
    
    k2 = model_fn(best_params, u_curr + 0.5 * dt * k1, trunk_inputs_test)   #(ns*nt, nx*ny)
    k2 = k2.reshape(k2.shape[0], Nx, Ny, Nz)    #(ns*nt, nx, ny)
    
    k3 = model_fn(best_params, u_curr + 0.5 * dt * k2, trunk_inputs_test)     #(ns*nt, nx*ny)
    k3 = k3.reshape(k3.shape[0], Nx, Ny, Nz)    #(ns*nt, nx, ny)
    
    k4 = model_fn(best_params, u_curr + dt * k3, trunk_inputs_test)    #(ns*nt, nx*ny)
    k4 = k4.reshape(k4.shape[0], Nx, Ny, Nz)    #(ns*nt, nx, ny)
    
    # Calculate the next state using RK4
    u_pred_next = u_curr + (dt / 6) * (k1 + 2 * k2 + 2 * k3 + k4)  #(ns*nt, nx, ny)
    
    return u_pred_next


def run_inference(initial_u, trunk_inputs_test, n_steps, method, dt):
    u_states = np.zeros(shape = (initial_u.shape[0], n_steps, Nx, Ny, Nz))  # List to store the states over time
    u_states[:,0,:,:,:] = initial_u
    
    # Initialize the previous state (this could be your u_0 and u_1, etc.)
    u_prev = initial_u  # Set the previous state to the initial state
    u_curr = initial_u  # Set the current state to the initial state
    
    for i in range(1, n_steps):
        
        if method == "AB":
            # Perform one inference step using the multistep method
            u_next = inference_ab(u_curr, u_prev, trunk_inputs_test, dt)
            
            #Set the first quadrant in the L-shaped to all zeros
            u_next = u_next.at[:, Nx//2:, Ny//2:, :].set(0.0)
            
            # Append the predicted state to the list
            u_states[:, i, :, :, :] = u_next

            # Update previous and current states for the next step
            u_prev = u_curr
            u_curr = u_next
        elif method == "RK":
            #Perform one inference step using the rk-4 method
            u_next = inference_rk(u_curr, trunk_inputs_test, dt)
            
            #Set the first quadrant in the L-shaped to all zeros
            u_next = u_next.at[:, Nx//2:, Ny//2:, :].set(0.0)
            
            # Append the predicted state to the list
            u_states[:, i, :, :, :] = u_next
            
            #Update the current state for the next step
            u_curr = u_next
    
    return u_states

In [8]:
Ns, Nt, Nx, Ny, Nz = outputs.shape
print(f"Ns: {Ns}, Nt: {Nt}, Nx: {Nx}, Ny: {Ny}, Nz: {Nz}")

import time
best_params = load_model_params(path = filepath, filename = filename)
print(f"Best model params loaded with {filename}")

method = "AB"
print(f"Inference scheme being used: {method}")

print("Compute the no. of steps for inference!")
NSTEPS = Nt
print(f"Inference dt: {dt_val}, nsteps: {NSTEPS}")

Ns: 1000, Nt: 101, Nx: 32, Ny: 32, Nz: 16
Best model params loaded with model_params_best_3538.pkl
Inference scheme being used: AB
Compute the no. of steps for inference!
Inference dt: 0.01, nsteps: 101


In [9]:
u_curr = outputs[:, 0, :, :, :]
start_time = time.time()
u_pred = run_inference(u_curr, trunk_inputs_test, n_steps=NSTEPS, method = method, dt=dt_val)
end_time = time.time()

In [10]:
u_pred.shape, outputs.shape

((1000, 101, 32, 32, 16), (1000, 101, 32, 32, 16))

In [11]:
u_pred.min(), u_pred.max(), outputs.min(), outputs.max()

(np.float64(-0.08262686431407928),
 np.float64(0.9990419358210508),
 np.float64(0.0),
 np.float64(0.9990419358210508))

In [12]:
print(f"Total time of inference for {u_pred.shape[0]} samples: ",end_time-start_time)
print(f"Prediction shape: {u_pred.shape}, Outputs shape: {outputs.shape}")

overall_rel_l2_err = jnp.linalg.norm(u_pred - outputs)/jnp.linalg.norm(outputs)
print(f"Overall relative L2 error: {overall_rel_l2_err}")

Total time of inference for 1000 samples:  11.348445415496826
Prediction shape: (1000, 101, 32, 32, 16), Outputs shape: (1000, 101, 32, 32, 16)
Overall relative L2 error: 0.018813833594322205


In [13]:
#Plotting the relative L2 error obtained at every timestep to show accummulation of autoregressive error
auto_reg_error = []
num_time_steps = Nt

for i in range(num_time_steps):
    l2_error = jnp.linalg.norm(u_pred[:,i,:,:] - outputs[:,i,:,:])/jnp.linalg.norm(outputs[:,i,:,:])
    auto_reg_error.append(l2_error)

#Compute statistics
# t = [10, 20, 30, 40, 50, 60, 70, 90, 100]
t = [48, 63, 93, 100]

print("----Extrapolation errors----")
for t_idx in t:
    print(f"t: {t_idx}, L2 error: {auto_reg_error[t_idx]}")

----Extrapolation errors----
t: 48, L2 error: 0.014513165690004826
t: 63, L2 error: 0.01979902759194374
t: 93, L2 error: 0.033520426601171494
t: 100, L2 error: 0.03718240186572075


In [14]:
save = True

if save:
    np.savez_compressed(filepath / f"results_{seed}_{method}.npz", 
                        u_pred=u_pred, 
                        u_actual=outputs)